<a href="https://colab.research.google.com/github/Sameekshaingole/fraud-detection-federated-learning/blob/main/Baseline_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

BASELINE MODEL (CENTRALIZED)

 Handles:

Class imbalance
Proper evaluation
Threshold tuning
Overfitting control
Feature importance

In [ ]:
# =========================
# BASELINE MODEL
# =========================

import time
import numpy as np
import pandas as pd
import pickle

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

# ---- Handle class imbalance ----
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights_dict = {0: class_weights[0], 1: class_weights[1]}

# ---- Train ----
start = time.time()

baseline_model = LogisticRegression(
    max_iter=1000,
    class_weight=class_weights_dict,
    solver='liblinear'
)

baseline_model.fit(X_train_scaled, y_train)

baseline_time = time.time() - start

# ---- Predict ----
y_pred_base = baseline_model.predict(X_test_scaled)
y_prob_base = baseline_model.predict_proba(X_test_scaled)[:,1]

# ---- Threshold tuning ----
threshold = 0.3
y_pred_base_tuned = (y_prob_base > threshold).astype(int)

# ---- Evaluation ----
print("\n🔥 BASELINE RESULTS 🔥")

print("Accuracy:", accuracy_score(y_test, y_pred_base_tuned))
print("Precision:", precision_score(y_test, y_pred_base_tuned))
print("Recall:", recall_score(y_test, y_pred_base_tuned))
print("F1:", f1_score(y_test, y_pred_base_tuned))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_base))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_base_tuned))
print("\nTraining Time:", baseline_time)

# ---- Save model ----
with open('/content/drive/MyDrive/fraud_detection_project/baseline_model.pkl', 'wb') as f:
    pickle.dump(baseline_model, f)